In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import dhg
import pdb
import torch.optim as optim
import matplotlib.pyplot as plt
import networkx as nx
import hypernetx as hnx
import random
import torch_geometric
from torch_scatter import scatter_mean, scatter_max, scatter_sum
from datetime import datetime

/workspaces/hello-hypergraphs/.venv/lib/python3.11/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /workspaces/hello-hypergraphs/.venv/lib/python3.11/site-packages/torch_scatter/_version_cpu.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/workspaces/hello-hypergraphs/.venv/lib/python3.11/site-packages/torch_geometric/llm/utils/backend_utils.py:26: DeprecationWarning: `torch_geometric.distributed` has been deprecated since 2.7.0 and will no longer be maintained. For distributed training, refer to our tutorials on distributed training at https://pytorch-geometric.readthedocs.io/en/latest/tutorial/distributed.html or cuGraph examples at https://github.com/rapidsai/cugraph-gnn/tree/main/python/cugraph-pyg/cugraph_pyg/examples
  from torch_geometric.distributed import (


OSError: /workspaces/hello-hypergraphs/.venv/lib/python3.11/site-packages/torch_scatter/_version_cpu.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE

In [ ]:
def gallery(graphs, labels=None, node_emb=None, special_color=False, max_graphs=4, max_fig_size=(40, 10), layout=nx.layout.kamada_kawai_layout):
    ''' Draw multiple graphs as a gallery
    Args:
      graphs: torch_geometrics.dataset object/ List of Graph objects
      labels: num_graphs
      node_emb: num_graphs* [num_nodes x num_ch]
      max_graphs: maximum graphs display
    '''
    num_graphs = min(len(graphs), max_graphs)
    ff, axes = plt.subplots(1, num_graphs,
                            figsize=max_fig_size,
                            subplot_kw={'xticks': [], 'yticks': []})
    if num_graphs == 1:
        axes = [axes]
    if node_emb is None:
        node_emb = num_graphs*[None]
    if labels is None:
        labels = num_graphs * [" "]


    for i in range(num_graphs):
        draw_one_graph(axes[i], graphs[i].edge_index.numpy(), labels[i], node_emb[i], layout, special_color)
        if labels[i] != " ":
            axes[i].set_title(f"Target: {labels[i]}", fontsize=28)
        axes[i].set_axis_off()
    plt.show()

In [ ]:
def draw_one_graph(ax, edges, label=None, node_emb=None, layout=None, special_color=False):
    """draw a graph with networkx based on adjacency matrix (edges)
    graph labels could be displayed as a title for each graph
    node_emb could be displayed in colors
    """
    graph = nx.Graph()
    edges = zip(edges[0], edges[1])
    graph.add_edges_from(edges)
    node_pos = layout(graph)
    #add colors according to node embeding
    if (node_emb is not None) or special_color:
        color_map = []
        node_list = [node[0] for node in graph.nodes(data = True)]
        for i,node in enumerate(node_list):
            #just ignore this branch
            if special_color:
                if len(node_list) == 3:
                    crt_color = (1,0,0)
                elif len(node_list) == 5:
                    crt_color = (0,1,0)
                elif len(node_list) == 4:
                    crt_color = (1,1,0)
                else:
                  special_list = [(1,0,0)] * 3 + [(0,1,0)] * 5 + [(1,1,0)] * 4
                  crt_color = special_list[i]
            else:
                crt_node_emb = node_emb[node]
                #map float number (node embeding) to a color
                crt_color = cm.gist_rainbow(crt_node_emb, bytes=True)
                crt_color = (crt_color[0]/255.0, crt_color[1]/255.0, crt_color[2]/255.0, crt_color[3]/255.0)
            color_map.append(crt_color)

        nx.draw_networkx_nodes(graph,node_pos, node_color=color_map,
                        nodelist = node_list, ax=ax)
        nx.draw_networkx_edges(graph, node_pos, ax=ax)
        nx.draw_networkx_labels(graph,node_pos, ax=ax)
    else:
        nx.draw_networkx(graph, node_pos, ax=ax)

In [ ]:
def update_stats(training_stats, epoch_stats):
    """ Store metrics along the training
    Args:
      epoch_stats: dict containg metrics about one epoch
      training_stats: dict containing lists of metrics along training
    Returns:
      updated training_stats
    """
    if training_stats is None:
        training_stats = {}
        for key in epoch_stats.keys():
            training_stats[key] = []
    for key,val in epoch_stats.items():
        training_stats[key].append(val)
    return training_stats

In [ ]:
def plot_stats(training_stats, figsize=(5, 5), name=""):
    """ Create one plot for each metric stored in training_stats
    """
    stats_names = [key[6:] for key in training_stats.keys() if key.startswith('train_')]
    f, ax = plt.subplots(len(stats_names), 1, figsize=figsize)
    if len(stats_names)==1:
        ax = np.array([ax])
    for key, axx in zip(stats_names, ax.reshape(-1,)):
        axx.plot(
            training_stats['epoch'],
            training_stats[f'train_{key}'],
            label=f"Training {key}")
        axx.plot(
            training_stats['epoch'],
            training_stats[f'val_{key}'],
            label=f"Validation {key}")
        axx.set_xlabel("Training epoch")
        axx.set_ylabel(key)
        axx.legend()
    plt.title(name)

In [ ]:
def get_list_of_edges(hypergraph):
  incidence_matrix = hypergraph.incidence_matrix()
  list_of_edges = []
  for i in range(incidence_matrix.shape[-1]):
    current_list = []
    for j in range(incidence_matrix.shape[0]):
      if incidence_matrix[j,i] != 0:
        current_list.append(j)
    list_of_edges.append(current_list)
  return list_of_edges

In [ ]:
def visualise(hypergraph):
  list_of_edges = get_list_of_edges(hypergraph)
  list_of_nodes = list(range(hypergraph.num_nodes))
  empty_dic = {}
  i=0
  for current_list in list_of_edges:
    empty_dic[str(i)] = current_list
    i+=1
  for current_node in list_of_nodes:
    empty_dic[str(i)] = [current_node]
    i+=1
  final_graph = hnx.Hypergraph(empty_dic)
  hnx.draw(final_graph)
  plt.title(f'Value = {hypergraph.y}')
  plt.show()

In [ ]:
def hnxHyperGraph(hypergraph):
  list_of_edges = get_list_of_edges(hypergraph)
  list_of_nodes = list(range(hypergraph.num_nodes))
  empty_dic = {}
  i=0
  for current_list in list_of_edges:
    empty_dic[str(i)] = current_list
    i+=1
  for current_node in list_of_nodes:
    empty_dic[str(i)] = [current_node]
    i+=1
  final_graph = hnx.Hypergraph(empty_dic)
  return final_graph

In [ ]:
def incidence_to_edgeindex(incidence_matrix):
  incidence_matrix = incidence_matrix.to_sparse()
  values = incidence_matrix.values()
  indices = incidence_matrix.indices()

  edge_list = []
  vertex_list = []
  for t in range(int(values.shape[0])):
    for j in range(int(values[t])):
      edge_list.append(indices[1,t])
      vertex_list.append(indices[0,t])
  edge_index = torch.tensor([vertex_list, edge_list])
  return edge_index

In [ ]:
class Graph(object):
    def __init__(self, edge_index, x, y, weighted = None):
        """ Graph structure
            for a mini-batch it will store a big (sparse) graph
            representing the entire batch
        Args:
            x: node features  [num_nodes x num_feats]
            y: graph labels   [num_graphs]
            edge_index: list of edges [2 x num_edges]
        """
        self.edge_index = edge_index
        self.x = x.to(torch.float32)
        self.y = y
        self.num_nodes = self.x.shape[0]
        if weighted is None:
          self.values = torch.ones(self.edge_index.shape[1])
        else:
          self.values = weighted

    #ignore this for now, it will be useful for batching
    def set_batch(self, batch):
        """ list of ints that maps each node to the graph it belongs to
            e.g. for batch = [0,0,0,1,1,1,1]: the first 3 nodes belong to graph_0 while
            the last 4 belong to graph_1
        """
        self.batch = batch

    # this function returns a sparse tensor
    def get_adjacency_matrix(self):
        """ from the list of edges create
        a num_nodes x num_nodes sparse adjacency matrix
        """
        return torch.sparse.LongTensor(self.edge_index,
                              # we work with a binary adj containing 1 if an edge exist
                              self.values,
                              torch.Size((self.num_nodes, self.num_nodes))
                              ).to_dense()

In [ ]:
class HyperGraph(object):
  def __init__(self, hyper_edge_index, x, y):
    """ HyperGraph structure
            for a mini-batch it will store a big hypergraph
            representing the entire batch
        Args:
            x: node features  [num_nodes x num_feats]
            y: hypergraph labels   [num_hyper_graphs]
            hyper_edge_index: list of hyperedges [2, k] where k is defined in
            the above cell.
        """
    self.hyper_edge_index = hyper_edge_index
    self.x = x.to(torch.float32)
    self.y = y
    self.num_nodes = self.x.shape[0] ##number of nodes
    self.num_hyper_edges = self.hyper_edge_index[1,:].max().item()+1 #number of hyper edges

  #this will be useful later, but we can ignore it for now
  def set_batch(self, batch):
    self.batch = batch


  #returns the incidence matrix H
  def incidence_matrix(self):
    return torch.sparse.LongTensor(self.hyper_edge_index,
                              # we work with a binary incidence containing 1 if an edge exist
                              torch.ones((self.hyper_edge_index.shape[1])),
                              torch.Size((self.num_nodes, self.num_hyper_edges))
                              ).to_dense()


In [ ]:
num_vertices = 6
num_edges = 3
num_features = 16
n=8
x = torch.rand((6,3))
y = 1
incidence_matrix = torch.concat((torch.randint(0, num_vertices, (1,n)), torch.randint(0, num_edges, (1,n))), dim = 0)
random_hypergraph = HyperGraph(incidence_matrix, x, y)

visualise(random_hypergraph)
print(f'The incidence matrix of this hypergraph is \n {random_hypergraph.incidence_matrix()}')

In [ ]:
random_hypergraph.hyper_edge_index

In [ ]:
dataset = dhg.data.CoauthorshipCora(data_root='')

In [ ]:
class CoCoraHyperDataset(object):
  def __init__(self, Dataset):
    """
    Dataset is a dhg object storing CoCora data. We will convert it to a
    hypergraph, graph, and weighted graph. Using clique and weighted clique
    expansion respectively (more on this later).
    """
    self.train_mask = Dataset['train_mask']
    self.val_mask = Dataset['val_mask']
    self.test_mask = Dataset['test_mask']
    self.features = Dataset['features']
    self.num_vertices = Dataset['features'].shape[0]
    self.labels = Dataset['labels']
    self.edge_list = Dataset['edge_list']

    ## Creating the hypergraph from the data
    self.edge_index = self.edge_list_2_index(self.edge_list)
    self.hyper_graph = HyperGraph(self.edge_index, self.features, self.labels)


    #creating a graph for the data
    self.graph_edge_index = self.edge_list_2_graph_index(self.edge_list)
    self.graph = Graph(self.graph_edge_index, self.features, self.labels)

    #creating a graph using weighted clique expansion
    self.wgraph_edge_index, self.wgraph_weights = self.edge_list_2_weighted_edge_index(self.edge_list, self.features.shape[0])
    self.wgraph = Graph(self.wgraph_edge_index, self.features, self.labels, self.wgraph_weights)

  def edge_list_2_index(self, edge_list):
    """
    Args
      edge_list : a list of lists each representing nodes forming hyperedge
    returns :
      hyper_edge_index : tensor containing indices of incidence matrix [2, k]
      where k is number of non-zero values in incidence matrix
    """
    index = 0
    hyper_vertices_list = []
    hyper_edges_list = []
    for i in edge_list:
      for j in range(len(i)):
        hyper_edges_list+=[index]
      hyper_vertices_list+=i
      index+=1
    hyper_edge_index = torch.tensor([hyper_vertices_list, hyper_edges_list])
    return hyper_edge_index

  def edge_list_2_graph_index(self, edge_list):
    """
    function that creates adjacency matrix associated to clique expansion of
    hypergraph
    Args
      edge_list : a list of lists each representing nodes forming hyperedge
    returns :
      edge_index: tensor containing indices of adjacency matrix [2, k] where k
      is number of non-zero values in adjacency matrix
    """
    source_list = []
    dest_list = []
    for current_edges in edge_list:
      # for every hyperedge draw an edge between every distinct pair of nodes
      n = len(current_edges)
      for i in range(n):
        for j in range(n):
          if i==j:
            continue
          source_list.append(current_edges[i])
          dest_list.append(current_edges[j])
    edge_index = torch.tensor([source_list, dest_list])
    return edge_index

  def edge_list_2_weighted_edge_index(self, edge_list, num_vertices):
    """
    function that creates adjacency matrix associated to weighted clique
    expansion of hypergraph
    Args:
      edge_list : a list of lists each representing nodes forming hyperedge
    returns :
      index: tensor containing indices of adjacency matrix [2, k] where k is
      number of non-zero values in adjacency matrix
      vals: weight associated to each edge
    """
    adjacency_matrix = torch.zeros((num_vertices, num_vertices))
    for current_edges in edge_list:
      # for every hyperedge draw an edge between every pair of nodes
      # for each pair of nodes the weight corresponds to number hyperedges they
      # have in common
      for i in current_edges:
        for j in current_edges:
          # this can be done with HH^T and subtracting degree matrix
          adjacency_matrix[i,j] += 1
    adjacency_matrix = adjacency_matrix.to_sparse()
    index = adjacency_matrix.indices()
    vals = adjacency_matrix.values()

    return index,vals

In [ ]:
Data = CoCoraHyperDataset(dataset)

In [ ]:
class HyperNNLayer(nn.Module):
  def __init__(self, input_dim, output_dim):
    """
    One layer of hypergraph neural network

    Args:
      input_dim : number of features of each node in hyergraph
      output_dim : number of output features
    """
    super(HyperNNLayer, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.Linear = nn.Linear(input_dim,output_dim)

  def forward(self, x,H):
    """
    Args:
      x : feature matrix [num_nodes, input_dim]
      H : incidence matrix [num_nodes, num_hyper_edges]
    returns:
      x : output of one layer of hypergraph neural network [num_nodes, output_dim]
    """
    # compute degree of nodes (D_v)^-0.5
    degree_of_nodes = torch.nan_to_num(torch.pow(torch.diag(torch.sum(H, dim=-1)), -0.5), nan=0, posinf=0, neginf=0).to(torch.float32)
    # compute degree of hyper edges (D_e)^-1
    degree_of_edges = torch.nan_to_num(torch.pow(torch.diag(torch.sum(H, dim=0)), -1.0), nan=0, posinf=0, neginf=0).to(torch.float32)


    # compute D_v^-0.5 H D_e^-1 H^T D_v^-0.5 x
    x = degree_of_nodes @ x
    x = torch.transpose(H, 0, 1) @ x
    x = degree_of_edges @ x
    x = H @ x
    x = degree_of_nodes @ x

    #apply linear layer
    x = self.Linear(x)
    return x

In [ ]:
class HyperNN(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim, num_layers):
    """
    Hypergraph neural network containing num_layers HyperNNLayer

    Args:
      input_dim : number of features of each node in hyergraph
      output_dim : number of output features
      hidden_dim : hidden dimension
      num_layers : number of layers
    """
    super(HyperNN, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.hidden_dim=hidden_dim

    if num_layers > 1:
      self.hnn_layers = [HyperNNLayer(input_dim, hidden_dim)]
      self.hnn_layers+= [HyperNNLayer(hidden_dim, hidden_dim) for i in range(num_layers-2)]
      self.hnn_layers+= [HyperNNLayer(hidden_dim, output_dim)]
    else:
      self.hnn_layers = [HyperNNLayer(input_dim, output_dim)]

    self.hnn_layers = nn.ModuleList(self.hnn_layers)
    self.num_layers = num_layers

  def forward(self,hgraph):
    """
    Args:
      hgraph : input hypergraph stored as HyperGraph class
    returns:
      y_hat : logits for each node [num_nodes, output_dim]
    """
    H = hgraph.incidence_matrix()

    x = hgraph.x.to(torch.float32)
    for j in range(self.num_layers-1):
      x = self.hnn_layers[j](x,H)
      x=F.relu(x)
    x = self.hnn_layers[self.num_layers-1](x, H)
    y_hat = x
    return y_hat

In [ ]:
def testing_hnn():
  torch.random.manual_seed(0)
  np.random.seed(0)

  input_dim = 64
  output_dim = 128
  #A = torch.tensor([[0, 1, 1, 0], [1, 0, 0, 0], [1, 1, 0, 1], [1, 1, 1, 0]])
  H = random_hypergraph.incidence_matrix()
  model = HyperNNLayer(input_dim, output_dim)

  x = torch.rand(H.shape[0], input_dim)
  out = model(x, H)

  assert(out.shape == (H.shape[0], output_dim)), "Oups! 🤭 Output shape is wrong"

  np.random.seed(0)
  perm_x = torch.tensor(np.random.permutation(x.numpy()))
  np.random.seed(0)
  perm_out = torch.tensor(np.random.permutation(out.detach().numpy()))

  np.random.seed(0)
  A_perm = np.random.permutation(H.detach().numpy().transpose()).transpose()
  np.random.seed(0)
  A_perm = torch.tensor(np.random.permutation(A_perm))

  torch.random.manual_seed(0)
  model_perm = HyperNNLayer(input_dim, output_dim)

  out_model_perm = model_perm(perm_x, A_perm)

  assert (torch.allclose(perm_out, out_model_perm, atol=1e-6)), "🤔 Something is wrong in the model! You are not permuation equivariant anymore 🥺"
  print("All good!")

testing_hnn()
np.random.seed(None)
torch.random.manual_seed(datetime.now().timestamp())

In [ ]:
NUM_EPOCHS = 50 #@param {type:"integer"}
LR = 0.001 #@param {type:"number"}
num_runs = 3

In [ ]:
def quick_accuracy(y_hat, y):
  """
  Args :
    y_hat : logits predicted by model [n, num_classes]
    y : ground trutch labels [n]
  returns :
    average accuracy
  """
  n = y.shape[0]
  y_hat = torch.argmax(y_hat, dim=-1)
  accuracy = (y_hat==y).sum().data.item()
  return accuracy/n

In [ ]:
def trainCoCora(hypergraph, model, mask, optimiser):
  model.train()
  y = hypergraph.y[mask]
  optimiser.zero_grad()
  y_hat = model(hypergraph)[mask] #only make predicitions for the ones we know the labels of
  loss = F.cross_entropy(y_hat, y)
  loss.backward()
  optimiser.step()
  return loss.data

In [ ]:
def evalCoCora(hypergraph, model, mask):
  model.eval()
  y = hypergraph.y[mask]
  y_hat = model(hypergraph)[mask]
  accuracy = quick_accuracy(y_hat, y)
  return accuracy

In [ ]:
def train_eval_loop_CoCora(model, hypergraph, train_mask,
                           valid_mask, test_mask):
    optimiser = optim.Adam(model.parameters(), lr=LR)
    training_stats = None
    # Training loop
    for epoch in range(NUM_EPOCHS):
        train_loss = trainCoCora(hypergraph,model, train_mask, optimiser)
        train_acc = evalCoCora(hypergraph, model,train_mask)
        valid_acc = evalCoCora(hypergraph, model, valid_mask)
        if epoch % 10 == 0:
            print(f"Epoch {epoch} with train loss: {train_loss:.3f} train accuracy: {train_acc:.3f} validation accuracy: {valid_acc:.3f}")
        # store the loss and the accuracy for the final plot
        epoch_stats = {'train_acc': train_acc, 'val_acc': valid_acc, 'epoch':epoch}
        training_stats = update_stats(training_stats, epoch_stats)
    # Lets look at our final test performance
    test_acc = evalCoCora(hypergraph, model, test_mask)
    print(f"Our final test accuracy for this model is: {test_acc:.3f}")
    return training_stats

In [ ]:
HyperNNModel = HyperNN(1433, 7, 128, 2)
# Data.hyper_graph stores our hypergraph structure
hyperNNModelOut = train_eval_loop_CoCora(HyperNNModel, Data.hyper_graph,
                                         Data.train_mask, Data.val_mask,
                                         Data.test_mask)
plot_stats(hyperNNModelOut)

In [ ]:
class Graph(object):
    def __init__(self, edge_index, x, y, weighted = None):
        """ Graph structure
            for a mini-batch it will store a big (sparse) graph
            representing the entire batch
        Args:
            x: node features  [num_nodes x num_feats]
            y: graph labels   [num_graphs]
            edge_index: list of edges [2 x num_edges]
        """
        self.edge_index = edge_index
        self.x = x.to(torch.float32)
        self.y = y
        self.num_nodes = self.x.shape[0]
        if weighted is None:
          self.values = torch.ones(self.edge_index.shape[1])
        else:
          self.values = weighted

    #ignore this for now, it will be useful for batching
    def set_batch(self, batch):
        """ list of ints that maps each node to the graph it belongs to
            e.g. for batch = [0,0,0,1,1,1,1]: the first 3 nodes belong to graph_0 while
            the last 4 belong to graph_1
        """
        self.batch = batch

    # this function returns a sparse tensor
    def get_adjacency_matrix(self):
        """ from the list of edges create
        a num_nodes x num_nodes sparse adjacency matrix
        """
        return torch.sparse.LongTensor(self.edge_index,
                              # we work with a binary adj containing 1 if an edge exist
                              self.values,
                              torch.Size((self.num_nodes, self.num_nodes))
                              ).to_dense()

In [ ]:
class GCNLayer(nn.Module):
  def __init__(self,input_dim, output_dim):
    super(GCNLayer, self).__init__()
    self.input_dim =input_dim
    self.output_dim = output_dim
    """GCN layer to be implemented by students of practical

    Args:
        input_dim (int): Dimensionality of the input feature vectors
        output_dim (int): Dimensionality of the output softmax distribution
        A (torch.Tensor): 2-D adjacency matrix
    """


    self.linear = nn.Linear(input_dim, output_dim)
        # =========================================

  def forward(self, x, A):

      D = torch.nan_to_num(torch.pow(torch.diag(torch.sum(A, dim=0)), -0.5), nan=0, posinf=0, neginf=0)
      self.adj_norm = D@A@D
      x = self.adj_norm@x
      x=self.linear(x)
      return x

In [ ]:
class GNN(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
    """
    Graph convolutional network containing num_layers GCNLayer

    Args:
      input_dim : number of features of each node in graph
      output_dim : number of output features
      hidden_dim : hidden dimension
      num_layers : number of layers
    """
    super(GNN, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    if num_layers > 1:
      self.gcn_layers = [GCNLayer(input_dim, hidden_dim)]
      self.gcn_layers += [GCNLayer(hidden_dim, hidden_dim) for i in range(num_layers-2)]
      self.gcn_layers += [GCNLayer(hidden_dim, output_dim)]
    else:
      self.gcn_layers = [GCNLayer(input_dim, output_dim)]

    self.gcn_layers = nn.ModuleList(self.gcn_layers)
    self.num_gcn_layers = num_layers

  def forward(self, graph):
    """
    Args:
      graph : input graph stored as Graph class
    returns:
      y_hat : logits for each node [num_nodes, output_dim]
    """
    x = graph.x.to(torch.float32)
    A = graph.get_adjacency_matrix()
    for j in range(self.num_gcn_layers-1):
      x = self.gcn_layers[j](x, A)
      x = F.relu(x)
    x = self.gcn_layers[-1](x,A)
    y_hat = x
    return y_hat

In [ ]:
graph_model = GNN(1433, 128, 7, 2)

# Data.graph stores clique expansion associated to hypergraph
graph_model_out = train_eval_loop_CoCora(graph_model, Data.graph, Data.train_mask, Data.val_mask, Data.test_mask)
plot_stats(graph_model_out)

In [ ]:
w_graph_model = GNN(1433, 128, 7, 2)
# Data.wgraph stores weighted clique expansion associated to hypergraph
graph_model_out = train_eval_loop_CoCora(w_graph_model, Data.wgraph, Data.train_mask, Data.val_mask, Data.test_mask)
plot_stats(graph_model_out)

In [ ]:
torch.random.manual_seed(0)

Hypergraph_Dataset = []
number_connected = 0
number_not_connected = 0
number_of_hgraphs = 1000
for j in range(number_of_hgraphs):
  num_vertices = torch.randint(5, 11, (1,1)).item() # select number of nodes uniformly between 5 and 11
  num_edges = torch.randint(1,10, (1,1)).item() # select number of hyper edges uniformly between 5 and 11
  nnz = torch.randint(num_edges, num_vertices*num_edges, (1,1)).item() # number of nnz in incidence matrix

  X = torch.rand((num_vertices, 6)) # randomly generate features
  y = torch.rand(1) # dummy label

  #randomly generate incidence matrix
  hyper_edge_index = torch.concat((torch.randint(0, num_vertices, (1,nnz)),
                                   torch.randint(0, num_edges, (1,nnz))), dim = 0)

  #create hypergraph as hnxHyperGraph to check connectivity
  hypergraph = hnxHyperGraph(HyperGraph(hyper_edge_index, X, y))

  #check if hypergraph is connected or not
  if hypergraph.is_connected() == True:
    number_connected +=1
    # assign label of 1 if connected
    Hypergraph_Dataset.append(HyperGraph(hyper_edge_index, X,  torch.tensor(1)))
  else:
    number_not_connected +=1
    # assign label of 0 if not connected
    Hypergraph_Dataset.append(HyperGraph(hyper_edge_index, X,  torch.tensor(0)))


print(f'Number of connected hypergraphs in our dataset = {number_connected} out of {1000} hypergraphs')

In [ ]:
visualise(Hypergraph_Dataset[10])
print(Hypergraph_Dataset[10].incidence_matrix().shape)

In [ ]:
# Hypergraph_Dataset

## We now need to split our dataset into testing, training and validation subset

train_data = Hypergraph_Dataset[0:700]
validation_data = Hypergraph_Dataset[700:850]
test_data = Hypergraph_Dataset[850:]

In [ ]:
def create_mini_batch(hgraph_list) -> Graph:
    """ Built a sparse graph from a batch of graphs
    Args:
        graph_list: list of Graph objects in a batch
    Returns:
        a big (sparse) Graph representing the entire batch
    """
    #insert first graph into the structure
    batch_edge_index = hgraph_list[0].hyper_edge_index
    batch_x = hgraph_list[0].x
    batch_y = [hgraph_list[0].y.item()]
    batch_batch = torch.zeros((hgraph_list[0].num_nodes), dtype=torch.int64)
    num_nodes=hgraph_list[0].num_nodes
    num_hyper_edges = hgraph_list[0].num_hyper_edges

    #append the rest of the graphs to the structure
    for idx, graph in enumerate(hgraph_list[1:]):
        # concat the features
        batch_x = torch.concat([batch_x, graph.x], dim=0)
        # concat the labels
        batch_y.append(graph.y.item())

        # concat the adjacency matrix as a block diagonal matrix
        current_hyper_edge_index = graph.hyper_edge_index[1,:] + num_hyper_edges
        current_node_index = graph.hyper_edge_index[0,:] + num_nodes
        current_index = torch.concat([current_node_index.unsqueeze(0), current_hyper_edge_index.unsqueeze(0)], dim=0)
        num_nodes+=graph.num_nodes
        num_hyper_edges+=graph.num_hyper_edges



        batch_edge_index = torch.concat([batch_edge_index, current_index], dim = 1)
        # ==========================================

        # create the array of indexes mapping nodes in the batch-graph
        # to the graph they belong to
        # specify the mapping between the new nodes and the graph they belong to (idx+1)
        batch_batch = torch.concat([batch_batch, (idx+1)*torch.ones([graph.num_nodes]).to(torch.int64)])
        # ==========================================
        pass

    #create the big sparse graph
    batch_graph = HyperGraph(batch_edge_index, batch_x, torch.tensor(batch_y))
    #attach the index array to the Graph structure
    batch_graph.set_batch(batch_batch)
    # print(batch_batch.dtype)
    return batch_graph

In [ ]:
smaller_list = Hypergraph_Dataset[0:3]
print(smaller_list[0].incidence_matrix().shape)
print(smaller_list[1].incidence_matrix().shape)
print(smaller_list[2].incidence_matrix().shape)

batch_graph = create_mini_batch(smaller_list)
visualise(batch_graph)
print(batch_graph.incidence_matrix().shape)

In [ ]:
class GHyperNNLayer(nn.Module):
  def __init__(self, input_dim, output_dim):
    """
    One layer of hypergraph neural network

    Args:
      input_dim : number of features of each node in hyergraph
      output_dim : number of output features
    """
    super(GHyperNNLayer, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.Linear = nn.Linear(input_dim,output_dim)

  def forward(self, x,H):
    """
    Args:
      x : feature matrix [num_nodes, input_dim]
      H : incidence matrix [num_nodes, num_hyper_edges]
    returns:
      x : output of one layer of hypergraph neural network [num_nodes, output_dim]
    """

    # compute degree of nodes (D_v)^-0.5
    degree_of_nodes = torch.nan_to_num(torch.pow(torch.diag(torch.sum(H, dim=-1)), -0.5), nan=0, posinf=0, neginf=0).to(torch.float32)
    # compute degree of hyper edges (D_e)^-1
    degree_of_edges = torch.nan_to_num(torch.pow(torch.diag(torch.sum(H, dim=0)), -1.0), nan=0, posinf=0, neginf=0).to(torch.float32)

    # compute D_v^-0.5 H D_e^-1 H^T D_v^-0.5 x
    x = degree_of_nodes @ x
    x = torch.transpose(H, 0, 1) @ x
    x = degree_of_edges @ x
    x = H @ x
    x = degree_of_nodes @ x

    #apply linear layer
    x = self.Linear(x)
    return x

In [ ]:
class GHyperNN(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim, num_layers):
    """
    Hypergraph neural network containing num_layers GHyperNNLayer for hypergraph
    level prediction

    Args:
      input_dim : number of features of each node in hyergraph
      output_dim : number of output features
      hidden_dim : hidden dimension
      num_layers : number of layers
    """
    super(GHyperNN, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.hidden_dim=hidden_dim

    if num_layers > 1:
      self.hnn_layers = [GHyperNNLayer(input_dim, hidden_dim)]
      self.hnn_layers+= [GHyperNNLayer(hidden_dim, hidden_dim) for i in range(num_layers-2)]
      self.hnn_layers+= [GHyperNNLayer(hidden_dim, output_dim)]
    else:
      self.hnn_layers = [GHyperNNLayer(input_dim, output_dim)]

    self.hnn_layers = nn.ModuleList(self.hnn_layers)
    self.num_layers = num_layers

  def forward(self,hgraph):
    """
    Args:
      hgraph : input hypergraph stored as HyperGraph class formed as a batch
    returns:
      y_hat : logits for each hypergraph in batch [batch_size, output_dim]
    """
    H = hgraph.incidence_matrix()
    x = hgraph.x.to(torch.float32)
    batch = hgraph.batch
    for j in range(self.num_layers-1):
      x = self.hnn_layers[j](x,H)
      x=F.relu(x)
    x = self.hnn_layers[self.num_layers-1](x, H)

    y_hat = scatter_mean(x, batch, dim=0)
    return y_hat

In [ ]:
def testing_hnn():
  torch.random.manual_seed(0)
  np.random.seed(0)

  input_dim = 6
  output_dim = 2
  hidden_dim = 128
  hypergraph = Hypergraph_Dataset[0]
  model = GHyperNN(input_dim, output_dim, hidden_dim, 3)
  # visualise(hypergraph)
  out = model(create_mini_batch([hypergraph]))

  assert(out.shape[-1] ==  output_dim), "Oups! 🤭 Output shape is wrong"

  np.random.seed(0)
  perm_x = torch.tensor(np.random.permutation(hypergraph.x.numpy()))


  H = hypergraph.incidence_matrix()


  np.random.seed(0)
  A_perm = torch.tensor(np.random.permutation(H.numpy()))
  A_perm = incidence_to_edgeindex(A_perm)
  perm_hypergraph = HyperGraph(A_perm, perm_x, hypergraph.y)
  torch.random.manual_seed(0)

  # visualise(perm_hypergraph)
  # visualise(hypergraph)
  out_model_perm = model(create_mini_batch([perm_hypergraph]))


  assert (torch.allclose(out, out_model_perm, atol=1e-4)), "🤔 Something is wrong in the model! You are not permuation invariant anymore 🥺"
  print("All good!")

testing_hnn()
np.random.seed(None)
torch.random.manual_seed(datetime.now().timestamp())

In [ ]:
BATCH_SIZE = 50 #@param {type:"integer"}
learning_rate = 0.001 #@param {type: "number"}
num_epochs = 10 #@param {type: "integer"}

In [ ]:
def train(dataset, model, optimiser, epoch, loss_fct, metric_fct, print_every = 50):
    """ Train model for one epoch
    """
    model.train()
    num_iter = int(len(dataset)/BATCH_SIZE)
    for i in range(num_iter):
        batch_list = dataset[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
        batch = create_mini_batch(batch_list)
        optimiser.zero_grad()
        y_hat= model(batch)
        loss = loss_fct(y_hat, batch.y)
        metric = metric_fct(y_hat, batch.y)
        loss.backward()
        optimiser.step()
        if (i+1) % print_every == 0:
          print(f"Epoch {epoch} Iter {i}/{num_iter}",
                    f"Loss train {loss}; Metric train {metric}")
    return loss, metric

In [ ]:
def evaluate(dataset, model, loss_fct, metrics_fct):
    """ Evaluate model on dataset
    """
    model.eval()
    # be careful in practice, as doing this way we will lose some
    # examples from the validation split, when len(dataset)%BATCH_SIZE != 0
    # think about how can you fix this!
    num_iter = int(len(dataset)/BATCH_SIZE)
    metrics_eval = 0
    loss_eval = 0
    for i in range(num_iter):
        batch_list = dataset[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
        batch = create_mini_batch(batch_list)


        y_hat = model(batch).to(torch.float32)

        metrics = metrics_fct(y_hat, batch.y)
        loss = loss_fct(y_hat, batch.y)

        metrics_eval += metrics
        loss_eval += loss.detach()
    metrics_eval /= num_iter
    loss_eval /= num_iter
    return loss_eval, metrics_eval

In [ ]:
def train_eval(model, train_dataset, val_dataset, test_dataset,
               loss_fct, metric_fct, print_every=1):
    """ Train the model for NUM_EPOCHS epochs
    """
    #Instantiatie our optimiser
    optimiser = optim.Adam(model.parameters(), lr=learning_rate)
    training_stats = None

    #initial evaluation (before training)
    val_loss, val_metric = evaluate(val_dataset, model, loss_fct, metric_fct)
    train_loss, train_metric = evaluate(train_dataset[:BATCH_SIZE], model,
                                        loss_fct, metric_fct)
    epoch_stats = {'train_loss': train_loss.detach(), 'val_loss': val_loss.detach(),
                      'train_metric': train_metric, 'val_metric': val_metric,
                      'epoch':0}
    training_stats = update_stats(training_stats, epoch_stats)

    for epoch in range(num_epochs):
        if isinstance(train_dataset, list):
            random.shuffle(train_dataset)
        else:
            train_dataset.shuffle()
        train_loss, train_metric = train(train_dataset, model, optimiser, epoch,
                                        loss_fct, metric_fct, print_every)
        val_loss, val_metric = evaluate(val_dataset, model, loss_fct, metric_fct)
        print(f"[Epoch {epoch+1}]",
                    f"train loss: {train_loss:.3f} val loss: {val_loss:.3f}",
                    f"train metric: {train_metric:.3f} val metric: {val_metric:.3f}"
              )
        # store the loss and the computed metric for the final plot
        epoch_stats = {'train_loss': train_loss.detach(), 'val_loss': val_loss.detach(),
                      'train_metric': train_metric, 'val_metric': val_metric,
                      'epoch':epoch+1}
        training_stats = update_stats(training_stats, epoch_stats)

    test_loss, test_metric = evaluate(test_dataset, model,  loss_fct, metric_fct)
    print(f"Test metric: {test_metric:.3f}")
    return training_stats

In [ ]:
model = GHyperNN(6, 2, 10, 3)

graph_level_out = train_eval(model, train_data, validation_data, test_data,loss_fct=F.cross_entropy, metric_fct=quick_accuracy, print_every=140)
plot_stats(graph_level_out)

In [ ]:
# Given 2 tensors a and b, we want to generate (a_i || b_j) for all (i,j) pairs
# a: [6, 5]
# b: [10, 7]
# broadcast_a_b -> [6, 10, 5+7]

a = torch.rand((6,5))
b = torch.rand((10,7))

# expand first tensor on the 2nd dimension and second tensor on the 1st dimension
a = a.unsqueeze(1) # a: [6,1,5]
b = b.unsqueeze(0) # b: [1,10,7]

# repeat the expanded dimensions to create tensors having the same dimension everywhere
# except from the dim where the concatenation will happen (last one in our case)
a = a.repeat(1, b.shape[1], 1) # a: [6,10,5]
b = b.repeat(a.shape[0], 1, 1) # b: [6,10,7]

# concatenate the 2 volumes on the last dimension
broadcast_a_b =  torch.concat((a, b), -1) # broadcast_a_b: [6,10,12]

print('Output shape is', broadcast_a_b.shape)

In [ ]:
class HyperGatLayer(nn.Module):
  def __init__(self, input_dim, output_dim):
    """
    One layer of hypergraph attention neural network

    Args:
      input_dim : number of features of each node in hyergraph
      output_dim : number of output features
    """
    super(HyperGatLayer, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.final_Linear = nn.Linear(input_dim, output_dim)
    self.mlp1 = nn.Linear(2*input_dim, 2*input_dim)
    self.mlp2 = nn.Linear(2*input_dim, 1)

  def forward(self, H, x):
    """
    Args:
      x : feature matrix [num_nodes, input_dim]
      H : incidence matrix [num_nodes, num_hyper_edges]
    returns:
      x : output of one layer of hypergraph neural network [num_nodes, output_dim]
    """
    num_nodes = x.shape[0]
    num_edges = H.shape[-1]

    # compute degree of nodes (D_v)^-0.5
    degree_of_nodes = torch.nan_to_num(torch.pow(torch.diag(torch.sum(H, dim=-1)), -0.5), nan=0, posinf=0, neginf=0).to(torch.float32)
    # compute degree of hyper edges (D_e)^-1
    degree_of_edges = torch.nan_to_num(torch.pow(torch.diag(torch.sum(H, dim=0)), -1.0), nan=0, posinf=0, neginf=0).to(torch.float32)

    # create features for each hyperedges and put in format for broadcasting
    edge_features = torch.transpose(H, 0, 1) @ x

    # reshape edge features  for broadcasting
    edge_features = edge_features.unsqueeze(0) # [1 x num_edges x input_dim]
    edge_features = edge_features.repeat(num_nodes, 1, 1) # [num_nodes x num_edges x input_dim]

    # put node features in format for broadcasting
    x_1 = x.unsqueeze(1) # [num_nodes x 1 x input_dim]
    x_1 = x_1.repeat(1, num_edges, 1) # [num_nodes x num_edges x input_dim]

    # concatenate node and hyperedge features
    concat_features = torch.concat((x_1, edge_features), -1)

    # apply MLP to obtain a scalar score for each(node, hedge pair)
    H_tilda = self.mlp2(F.relu(self.mlp1(concat_features))).squeeze(-1)
    # mask the weighted incidence matrix with the original H
    H_tilda = F.sigmoid(H_tilda)*H

    # compute the product replacing H with H_tilda and apply linear layer
    product = degree_of_nodes @ H_tilda @ degree_of_edges @ torch.transpose(H_tilda, 0,1) @degree_of_nodes
    projection = self.final_Linear(x)
    x = product @ projection
    return x

In [ ]:
class HyperGat(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim, num_layers):
    """
    Hypergraph neural network containing num_layers GHyperNNLayer for hypergraph
    level prediction

    Args:
      input_dim : number of features of each node in hyergraph
      output_dim : number of output features
      hidden_dim : hidden dimension
      num_layers : number of layers
    """
    super(HyperGat, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.hidden_dim=hidden_dim

    if num_layers > 1:
      self.hnn_layers = [HyperGatLayer(input_dim, hidden_dim)]
      self.hnn_layers+= [HyperGatLayer(hidden_dim, hidden_dim) for i in range(num_layers-2)]
      self.hnn_layers+= [HyperGatLayer(hidden_dim, output_dim)]
    else:
      self.hnn_layers = [HyperGatLayer(input_dim, output_dim)]

    self.hnn_layers = nn.ModuleList(self.hnn_layers)
    self.num_layers = num_layers

  def forward(self,hgraph):
    """
    Args:
      hgraph : input hypergraph stored as HyperGraph class formed as a batch
    returns:
      y_hat : logits for each hypergraph in batch [batch_size, output_dim]
    """
    x = hgraph.x
    y = hgraph.y
    H = hgraph.incidence_matrix()
    batch = hgraph.batch

    for j in range(self.num_layers-1):
      x = self.hnn_layers[j](H, x)
      x = F.relu(x)
    x = self.hnn_layers[-1](H,x)
    y_hat = scatter_mean(x, batch, dim=0)
    return y_hat

In [ ]:
def testing_hnn():
  torch.random.manual_seed(0)
  np.random.seed(0)

  input_dim = 6
  output_dim = 2
  hidden_dim = 128
  hypergraph = Hypergraph_Dataset[0]
  model = HyperGat(input_dim, output_dim, hidden_dim, 3)
  # visualise(hypergraph)
  out = model(create_mini_batch([hypergraph]))

  assert(out.shape[-1] ==  output_dim), "Oups! 🤭 Output shape is wrong"

  np.random.seed(0)
  perm_x = torch.tensor(np.random.permutation(hypergraph.x.numpy()))


  H = hypergraph.incidence_matrix()


  np.random.seed(0)
  A_perm = torch.tensor(np.random.permutation(H.numpy()))
  A_perm = incidence_to_edgeindex(A_perm)
  perm_hypergraph = HyperGraph(A_perm, perm_x, hypergraph.y)
  torch.random.manual_seed(0)

  # visualise(perm_hypergraph)
  # visualise(hypergraph)
  out_model_perm = model(create_mini_batch([perm_hypergraph]))


  assert (torch.allclose(out, out_model_perm, atol=1e-4)), "🤔 Something is wrong in the model! You are not permuation invariant anymore 🥺"
  print("All good!")

testing_hnn()
np.random.seed(None)
torch.random.manual_seed(datetime.now().timestamp())

In [ ]:
attention_model = HyperGat(6,2, 8, 3)
attention_model_out = train_eval(attention_model, train_data, validation_data,
                                 test_data,loss_fct=F.cross_entropy,
                                 metric_fct=quick_accuracy, print_every=140)
plot_stats(attention_model_out)